# `%%ai` 매직 + Jupyternaut 사용 데모

## LiteLLM 기반 `%%ai` 커스텀 매직

`jupyter-ai-magics`는 현재 환경의 `langchain 1.x`와 충돌하므로,
이미 설치된 `litellm`을 직접 wrapping한 `%%ai` 매직을 사용합니다.

| 호출 경로 | 형식 | 환경변수 |
|-----------|------|----------|
| Chat UI (`@Jupyternaut`) | `openrouter/<model>` | `OPENROUTER_API_KEY` |
| 노트북 셀 (`%%ai`) | `openrouter/<model>` | `OPENROUTER_API_KEY` |

두 경로 모두 같은 모델 ID 형식과 같은 환경변수를 사용합니다.

In [ ]:
from IPython.core.magic import register_cell_magic
from litellm import completion
from IPython.display import display, Markdown
import os

@register_cell_magic
def ai(line, cell):
    """%%ai <openrouter/model> [-f markdown|code|text]"""
    parts = line.strip().split()
    model = parts[0] if parts else 'openrouter/nvidia/nemotron-3-super-120b-a12b'
    fmt = parts[parts.index('-f') + 1] if '-f' in parts else 'markdown'
    resp = completion(
        model=model,
        messages=[{'role': 'user', 'content': cell}],
        api_key=os.environ.get('OPENROUTER_API_KEY'),
    )
    text = resp.choices[0].message.content
    display(Markdown(text) if fmt in ('markdown', 'md') else text)

NEMO = 'openrouter/nvidia/nemotron-3-super-120b-a12b'
print('%%ai 매직 등록 완료 — 사용법: %%ai openrouter/<model> [-f markdown|code|text]')
print(f'기본 모델 단축: NEMO = openrouter/nvidia/nemotron-3-super-120b-a12b')


## Step 1. 기본 호출

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b
Android 커널 패닉(kernel panic)이 발생하는 주요 원인 3가지를 간결하게 설명해줘.

## Step 2. 출력 포맷 (`-f`)

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f code
Python으로 로그 파일에서 'panic' 키워드가 포함된 줄만 출력하는 함수를 작성해줘.

## Step 3. Python 변수 보간

`{변수명}` 형식으로 현재 셀 환경의 Python 변수를 프롬프트에 주입합니다.

In [ ]:
sample_findings = {
    'panic_signals': ['kernel BUG at mm/slab.c:2887', 'Unable to handle kernel NULL pointer'],
    'suspicious_processes': ['kworker/2:1H', 'mali_jd_worker'],
    'network_anomalies': ['unexpected socket state: CLOSE_WAIT x47']
}
print(sample_findings)

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
아래 Android ramdump 분석 결과에서 root cause와 다음 조사 방향을 제안해줘.

{sample_findings}

## Step 4. Chat UI (Jupyternaut) 연동

노트북 분석 결과를 Chat UI에서 이어받는 방법:

```
@Jupyternaut @file:notebooks/interactive_log_analyzer.ipynb 분석 결과 요약해줘
@Jupyternaut 방금 분석한 dump의 suspicious_processes 중 가장 위험한 것은?
```

Jupyternaut의 `get_active_cell_id` 도구가 현재 선택된 셀을 자동으로 참조합니다.